In [1]:
# RoBERTa Model Training and Evaluation
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering, get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("/content/drive/MyDrive/PHD Folder/QA Model and Dataset/AmMedQA.csv")[:3000]
df = df.dropna(subset=['question', 'context', 'answer']).reset_index(drop=True)

def find_answer_start(row):
    if "Answer_start" in row and not pd.isna(row['Answer_start']):
        return int(row['Answer_start'])
    # Fallback to string search if offset isn't explicitly provided
    start_idx = str(row['context']).find(str(row['answer']))
    return start_idx

df['Answer_start'] = df.apply(find_answer_start, axis=1)

# Filter out examples where the answer span cannot be located in context
df = df[df['Answer_start'] != -1].reset_index(drop=True)

# Train/Test Split (80/20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train Samples: {len(train_df)}, Test samples: {len(test_df)}")

# Tokenization & Dataset Preparation
model_name = "rasyosef/roberta-base-amharic"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class AmharicQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=384, doc_stride=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.doc_stride = doc_stride

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        question = str(row['question'])
        context = str(row['context'])
        answer = str(row['answer'])
        start_char = int(row['Answer_start'])
        end_char = start_char + len(answer)

        # Tokenize Question and context
        inputs = self.tokenizer(
            question,
            context,
            max_length=self.max_len,
            truncation="only_second",
            return_offsets_mapping=True,
            padding="max_length",
            return_tensors="pt"
        )

        offset_mapping = inputs['offset_mapping'][0]
        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        # Find token start and end positions corresponding to character offsets
        sequence_ids = inputs.sequence_ids(0)

        # Find context bounds inside tokenized sequence (context tokens have sequence_id == 1)
        idx = 0
        while idx < len(sequence_ids) and sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Check if answer is within context bounds
        if context_start >= len(sequence_ids) or context_end < context_start or not (
            offset_mapping[context_start][0] <= start_char and offset_mapping[context_end][1] >= end_char
        ):
            start_token_idx = 0
            end_token_idx = 0
        else:
            # Locate token start
            idx = context_start
            while idx <= context_end and offset_mapping[idx][0] <= start_char:
                idx += 1
            start_token_idx = idx - 1

            # Locate token end
            idx = context_end
            while idx >= context_start and offset_mapping[idx][1] >= end_char:
                idx -= 1
            end_token_idx = idx + 1

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "start_positions": torch.tensor(start_token_idx, dtype=torch.long),
            "end_positions": torch.tensor(end_token_idx, dtype=torch.long),
            "context": context,
            "answer": answer,
            "offset_mapping": offset_mapping
        }

train_dataset = AmharicQADataset(train_df, tokenizer)
test_dataset = AmharicQADataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Model Training Setup
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.to(device)

epochs = 10
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Training Loop
print("\n--- Starting Model Training ---")
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        start_positions = batch['start_positions'].to(device)
        end_positions = batch['end_positions'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            start_positions=start_positions,
            end_positions=end_positions
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_train_loss:.4f}")

# Evaluation Metrics (EM and F1)
def compute_exact_match(prediction, ground_truth):
    return int(prediction.strip() == ground_truth.strip())

def compute_f1(prediction, ground_truth):
    pred_tokens = prediction.strip().split()
    truth_tokens = ground_truth.strip().split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    common_tokens = set(pred_tokens) & set(truth_tokens)
    if len(common_tokens) == 0:
        return 0.0
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

# Model Evaluation
print("\n--- Evaluating RoBERTa on Test Set ---")
model.eval()
exact_match_scores = []
f1_scores = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        for i in range(len(input_ids)):
            start_idx = torch.argmax(start_logits[i]).item()
            end_idx = torch.argmax(end_logits[i]).item()
            offsets = batch['offset_mapping'][i]
            context = batch['context'][i]
            ground_truth = batch['answer'][i]

            # Decode extracted answer span from character offsets
            if start_idx <= end_idx and end_idx < len(offsets):
                start_char = offsets[start_idx][0].item()
                end_char = offsets[end_idx][1].item()
                predicted_answer = context[start_char:end_char]
            else:
                predicted_answer = ""

            em = compute_exact_match(predicted_answer, ground_truth)
            f1 = compute_f1(predicted_answer, ground_truth)
            exact_match_scores.append(em)
            f1_scores.append(f1)

# Final Results
mean_em = np.mean(exact_match_scores) * 100
mean_f1 = np.mean(f1_scores) * 100

print("\n" + "="*40)
print(f"RoBERTa Evaluation Results ({model_name})")
print("="*40)
print(f"Exact Match (EM): {mean_em:.2f}%")
print(f"F1 Score: {mean_f1:.2f}%")
print("="*40)

Train Samples: 2296, Test samples: 574


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

Epoch 9 - Average Loss: 0.0654


Epoch 10/10: 100%|██████████| 144/144 [02:50<00:00,  1.18s/it]


Epoch 10 - Average Loss: 0.0566

--- Evaluating RoBERTa on Test Set ---


Evaluating: 100%|██████████| 36/36 [00:13<00:00,  2.72it/s]


RoBERTa Evaluation Results (rasyosef/roberta-base-amharic)
Exact Match (EM): 58.01%
F1 Score: 59.88%
